# LatentSR - AdaGN on VAE-SR (Kaggle)

Train the **spatial-FiLM conditioner** (`condition_type: adagn`) with the **frozen Q2 VAE-SR**. Same 50-epoch recipe as concat Q2. Do **not** resume a concat / Phase-8 checkpoint.

### Before Run All
1. Settings -> Accelerator: **GPU T4** if you are starting a **new** session (works with Kaggle's default PyTorch).
2. If this session is already on a **P100**, stay on P100. Switching GPU wipes `/kaggle/working` (CelebA). Run the PyTorch-compat cell instead.
3. Settings -> Internet: **ON**
4. Secret `HF_TOKEN` (write access to `HusseinHamouda/LatentSR-checkpoints`)

HF writes go to `latent_sr_adagn_q2/` and will not overwrite concat Q2.

A 404 for `latent_sr_adagn_q2/latest.pt` on the first run is expected (nothing uploaded yet). Training then starts from scratch.

If this session dies, **Run All** again: CelebA and VAE download skip if present, and training resumes from local `latest.pt` or HF `latent_sr_adagn_q2/latest.pt`.


## 1. Hugging Face login


In [ ]:
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"])

from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami, HfApi

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
login(token=HF_TOKEN)

print("HF account:", whoami()["name"])
api = HfApi()
api.repo_info(repo_id="HusseinHamouda/LatentSR-checkpoints", repo_type="model")
print("Repository access: OK")


## 2. Clone LatentSR + generative-models


In [ ]:
from pathlib import Path
import subprocess
import sys

def run(cmd, cwd=None):
    print("+", " ".join(cmd) if isinstance(cmd, list) else cmd)
    subprocess.check_call(cmd, cwd=cwd)


root = Path("/kaggle/working")
latentsr = root / "LatentSR"
gm = root / "generative-models"

if latentsr.exists():
    run(["git", "-C", str(latentsr), "pull"])
else:
    run(["git", "clone", "https://github.com/HusseinHanafy207/LatentSR.git", str(latentsr)])

if gm.exists():
    run(["git", "-C", str(gm), "pull"])
else:
    run(["git", "clone", "https://github.com/HusseinHanafy207/generative-models.git", str(gm)])

py = sys.executable
run([py, "-m", "pip", "install", "-q", "-e", str(gm)])
run([py, "-m", "pip", "install", "-q", "-e", str(latentsr)])
run([py, "-m", "pip", "install", "-q", "gdown", "huggingface_hub"])
print("install done")


## 2b. PyTorch vs GPU

Kaggle's default PyTorch (`cu128`) dropped Pascal. **T4 (sm_75)** is fine. **P100 (sm_60)** needs the CUDA 12.6 wheel or interpolate/training dies with `no kernel image is available`.

This cell is a no-op on T4. On P100 it reinstalls `torch`/`torchvision` from `cu126` (a few minutes). Training runs in a subprocess, so you do **not** need to restart the kernel.


In [ ]:
import subprocess
import sys

py = sys.executable


def run(cmd):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)


info = subprocess.check_output(
    [
        py,
        "-c",
        (
            "import torch; "
            "print(torch.__version__); "
            "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'); "
            "print(','.join(str(x) for x in (torch.cuda.get_device_capability(0) if torch.cuda.is_available() else (0,0)))); "
            "print(','.join(torch.cuda.get_arch_list()) if torch.cuda.is_available() else '')"
        ),
    ],
    text=True,
).strip().splitlines()
version, name, cap, archs = (info + ["", "", "0,0", ""])[:4]
major, minor = (int(x) for x in cap.split(","))
print(f"torch {version}")
print(f"GPU {name}  capability {major}.{minor}")
print(f"arch list: {archs}")

if not name or name == "cpu":
    raise RuntimeError("No CUDA GPU. Settings -> Accelerator -> GPU T4 (or P100).")

# Pascal (P100) is sm_60. Current Kaggle cu128 wheels start at sm_70.
if major < 7:
    print("P100 detected: installing PyTorch cu126 (keeps sm_60 kernels).")
    run([py, "-m", "pip", "uninstall", "-y", "torch", "torchvision", "torchaudio"])
    run(
        [
            py,
            "-m",
            "pip",
            "install",
            "--no-cache-dir",
            "torch==2.10.0",
            "torchvision",
            "torchaudio",
            "--index-url",
            "https://download.pytorch.org/whl/cu126",
        ]
    )
else:
    print("GPU is sm_70+; keeping Kaggle's default PyTorch.")

smoke = subprocess.check_output(
    [
        py,
        "-c",
        (
            "import torch, torch.nn.functional as F; "
            "x = torch.zeros(1, 3, 32, 32, device='cuda'); "
            "y = F.interpolate(x, size=128, mode='bicubic', align_corners=False); "
            "print(torch.__version__, torch.cuda.get_arch_list(), tuple(y.shape))"
        ),
    ],
    text=True,
).strip()
print("CUDA interpolate smoke test:", smoke)


## 3. CelebA (Drive copies)

Torchvision cannot download CelebA on Kaggle. These are the same Drive file IDs from the previous phases. Re-running skips files that already exist.


In [ ]:
from pathlib import Path
import subprocess

celeba = Path("/kaggle/working/data/raw/celeba")
celeba.mkdir(parents=True, exist_ok=True)
img_dir = celeba / "img_align_celeba"

# Replace only if a Drive link rotates.
files = {
    "img_align_celeba.zip": "1lVqCbFGvz_zEwFwGXZDE57fZgzyE54lX",
    "list_attr_celeba.txt": "1--ygZyBF_NVgZV0ghGdEKKw-RDT_CUsf",
    "list_eval_partition.txt": "1sfBkt6LULyQYBfEQenI-X2LcCaoiZkwk",
    "identity_CelebA.txt": "1_c8h_buw8wnTW_cfwk2KfpY-i6dUaKPH",
    "list_bbox_celeba.txt": "1KShnIpocgfBlBJ-2IUPAfEDJyOpvKF_c",
    "list_landmarks_align_celeba.txt": "1K_ycTaSHqhyiIfBohSPMDUXKdp0Dezpq",
}


def jpg_count(path: Path) -> int:
    if not path.exists():
        return 0
    return sum(1 for p in path.iterdir() if p.suffix.lower() == ".jpg")


n_img = jpg_count(img_dir)
if n_img >= 200000:
    print(f"CelebA already extracted ({n_img} jpg). Skip download.")
else:
    for name, file_id in files.items():
        dest = celeba / name
        if dest.exists() and dest.stat().st_size > 0:
            print("exists", dest.name)
            continue
        url = f"https://drive.google.com/uc?id={file_id}"
        subprocess.check_call(["gdown", url, "-O", str(dest)])
    zip_path = celeba / "img_align_celeba.zip"
    print("unzipping", zip_path)
    subprocess.check_call(["unzip", "-q", "-o", str(zip_path), "-d", str(celeba)])
    n_img = jpg_count(img_dir)
    print("jpg count:", n_img)
    if n_img < 200000:
        raise RuntimeError(f"expected ~202599 faces, got {n_img}")


## 4. Frozen VAE-SR (HF) - not VAE-1, not a DDPM ckpt


In [ ]:
from pathlib import Path
import shutil
import torch
from huggingface_hub import hf_hub_download

vae_dest = Path("/kaggle/working/outputs/vae_sr/checkpoints/latest.pt")
vae_dest.parent.mkdir(parents=True, exist_ok=True)

src = Path(
    hf_hub_download(
        repo_id="HusseinHamouda/LatentSR-checkpoints",
        filename="vae_sr/latest.pt",
        local_dir="/kaggle/working/hf_ckpt",
    )
)
if src.resolve() != vae_dest.resolve():
    shutil.copy2(src, vae_dest)

ckpt = torch.load(vae_dest, map_location="cpu", weights_only=False)
print("VAE-SR path:", vae_dest)
print("VAE-SR epoch:", ckpt.get("epoch"))
print("keys:", sorted(ckpt.keys())[:20])
if "model_state_dict" not in ckpt:
    raise RuntimeError("VAE checkpoint missing model_state_dict")


## 5. Confirm AdaGN config (do not concat)


In [ ]:
from pathlib import Path
import yaml

cfg_path = Path("/kaggle/working/LatentSR/configs/latent_sr_adagn_q2.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
print("config:", cfg_path)
print("condition_type:", cfg.get("condition_type"))
print("vae_checkpoint:", cfg.get("vae_checkpoint"))
print("checkpoint_dir:", cfg.get("checkpoint_dir"))
print("hf_checkpoint_subdir:", cfg.get("hf_checkpoint_subdir"))
if cfg.get("condition_type") not in {"adagn", "film"}:
    raise RuntimeError(f"expected adagn, got {cfg.get('condition_type')!r}")

out = Path("/kaggle/working/outputs/latent_sr_adagn_q2")
for sub in ("checkpoints", "samples", "logs"):
    (out / sub).mkdir(parents=True, exist_ok=True)
print("output dirs ready")


## 6. Train 50 epochs

Set `EPOCHS = 1` for a smoke test, then re-run this cell with `EPOCHS = 50`. If `latest.pt` exists locally or on HF under `latent_sr_adagn_q2/`, this cell resumes. It **refuses** a concat checkpoint.


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch
from huggingface_hub import hf_hub_download

EPOCHS = 50  # 1 = smoke test only
CONFIG = "/kaggle/working/LatentSR/configs/latent_sr_adagn_q2.yaml"
VAE = "/kaggle/working/outputs/vae_sr/checkpoints/latest.pt"
LOCAL = Path("/kaggle/working/outputs/latent_sr_adagn_q2/checkpoints/latest.pt")
REPO = "HusseinHamouda/LatentSR-checkpoints"


def condition_type_of(path: Path) -> str:
    ckpt = torch.load(path, map_location="cpu", weights_only=False)
    ctype = (ckpt.get("config") or {}).get("condition_type", "concat")
    print(f"{path}: epoch={ckpt.get('epoch')} condition_type={ctype}")
    if ctype not in {"adagn", "film"}:
        raise RuntimeError(
            f"Refusing to resume {path} (condition_type={ctype}). "
            "AdaGN cannot load a concat checkpoint."
        )
    return ctype


resume = None
if LOCAL.exists():
    condition_type_of(LOCAL)
    resume = str(LOCAL)
else:
    try:
        hf_path = Path(
            hf_hub_download(
                repo_id=REPO,
                filename="latent_sr_adagn_q2/latest.pt",
                local_dir="/kaggle/working/hf_ckpt",
            )
        )
        condition_type_of(hf_path)
        resume = str(hf_path)
    except Exception as exc:
        msg = str(exc)
        if "404" in msg or "Entry Not Found" in msg or "EntryNotFound" in type(exc).__name__:
            print("No AdaGN checkpoint on HF yet (first run). Training from scratch.")
        else:
            raise

cmd = [
    sys.executable,
    "scripts/train_sr.py",
    "--config",
    CONFIG,
    "--vae-checkpoint",
    VAE,
    "--epochs",
    str(EPOCHS),
    "--device",
    "cuda",
    "--no-download",
]
if resume:
    cmd += ["--resume", resume]

print("+", " ".join(cmd))
subprocess.check_call(cmd, cwd="/kaggle/working/LatentSR")
